# 01 — Cricket Filtering

This notebook filters the full Reddit dataset down to cricket subreddits only and saves the result back to S3. All later notebooks read from this filtered dataset.

### 1. Setup

Import utilities and start Spark.

In [1]:
# Add parent folder to path
import sys
sys.path.insert(0, "..")

# Import helpers
from utils import (
    get_spark, load_submissions, load_comments,
    CRICKET_SUBREDDITS,
    S3_CRICKET_SUBMISSIONS, S3_CRICKET_COMMENTS,
    save_to_s3,
)

# Spark functions
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lower, count


### 2. Start Spark

Build the Spark session.

In [2]:
# Start Spark
spark = get_spark("01_CricketFiltering")
spark


:: loading settings :: url = jar:file:/home/ubuntu/spark-3.5.1-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ubuntu/.ivy2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5d970426-6440-4a16-9308-81e6a4ae47fc;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 404ms :: artifacts dl 20ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evi

### 3. Show subreddit list

Print the cricket subreddits we will filter to.

In [3]:
# Print the target list
print(f"Filtering to {len(CRICKET_SUBREDDITS)} cricket subreddits:")
for s in CRICKET_SUBREDDITS:
    print(f"  - {s}")


Filtering to 46 cricket subreddits:
  - Cricket
  - CricketShitpost
  - IndianCricket
  - IndianCricketTeam
  - bcci
  - cricketindia
  - indiacricket
  - ipl
  - ChennaiSuperKings
  - csk
  - RoyalChallengers
  - RCB
  - MumbaiIndians
  - mi
  - KolkataKnightRiders
  - KKR
  - SunrisersHyderabad
  - srh
  - DelhiCapitals
  - delhicapitals
  - dc
  - PunjabKingsIPL
  - pbks
  - RajasthanRoyals
  - rr
  - LucknowSuperGiants
  - lsg
  - GujaratTitans
  - gt
  - Dhoni
  - MSDhoni
  - Kohli
  - ViratKohli
  - Virat
  - RohitSharma
  - Hitman
  - PakCricket
  - AusCricket
  - cricketaustralia
  - EngCricket
  - englandcricket
  - NZCricket
  - TestCricket
  - Ashes
  - ICC
  - t20worldcup


### 4. Load full submissions

Load the entire submissions dataset from S3.

In [4]:
# Load submissions
subs = load_submissions(spark)


26/04/28 17:35:49 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


### 5. Load full comments

Load the entire comments dataset from S3.

In [5]:
# Load comments
coms = load_comments(spark)


### 6. Filter submissions to cricket subreddits

Keep only rows where the subreddit is in our list (case-insensitive).

In [6]:
# Lowercase the target list for case-insensitive matching
lower_subs = [s.lower() for s in CRICKET_SUBREDDITS]

# Filter submissions
cricket_subs = subs.filter(lower(col("subreddit")).isin(lower_subs))


### 7. Filter comments to cricket subreddits

Same filter applied to comments.

In [7]:
# Filter comments
cricket_coms = coms.filter(lower(col("subreddit")).isin(lower_subs))


### 8. Save filtered submissions to S3

Write the filtered submissions back to S3 as parquet, partitioned by yyyy/mm.

In [ ]:
# Save filtered submissions, partitioned by yyyy/mm for fast reads later
save_to_s3(
    cricket_subs,
    S3_CRICKET_SUBMISSIONS,
    mode="overwrite",
    partition_by=["yyyy", "mm"],
)


### 9. Save filtered comments to S3

Write the filtered comments back to S3 as parquet, partitioned by yyyy/mm.

In [10]:
# Save filtered comments
save_to_s3(
    cricket_coms,
    S3_CRICKET_COMMENTS,
    mode="overwrite",
    partition_by=["yyyy", "mm"],
)


ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/home/ubuntu/cricket_project/cricket-env/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ubuntu/cricket_project/cricket-env/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

### 10. Verify filtered submissions

Read back the filtered submissions and count rows.

In [8]:
# Read back and count
verify_subs = spark.read.parquet(S3_CRICKET_SUBMISSIONS)
sub_count = verify_subs.count()
print(f"Filtered submissions saved: {sub_count:,}")


Filtered submissions saved: 242,886


### 11. Verify filtered comments

Read back the filtered comments and count rows.

In [9]:
# Read back and count
verify_coms = spark.read.parquet(S3_CRICKET_COMMENTS)
com_count = verify_coms.count()
print(f"Filtered comments saved: {com_count:,}")


Filtered comments saved: 3,217,571


### 12. Submissions per subreddit

Check post counts in the filtered submissions.

In [10]:
# Post counts per subreddit
verify_subs.groupBy("subreddit") \
    .agg(count("*").alias("post_count")) \
    .orderBy(col("post_count").desc()) \
    .show(50, truncate=False)


+-------------------+----------+
|subreddit          |post_count|
+-------------------+----------+
|Cricket            |65562     |
|CricketShitpost    |47453     |
|IndiaCricket       |40469     |
|ipl                |20812     |
|HiTMAN             |20191     |
|PakCricket         |9744      |
|RCB                |9416      |
|KolkataKnightRiders|8030      |
|csk                |5016      |
|MumbaiIndians      |4405      |
|SunrisersHyderabad |4320      |
|RajasthanRoyals    |2540      |
|delhicapitals      |2522      |
|EnglandCricket     |1060      |
|ViratKohli         |993       |
|RohitSharma        |135       |
|IndianCricket      |85        |
|srh                |58        |
|Dhoni              |30        |
|icc                |28        |
|Virat              |7         |
|lsg                |5         |
|BCCI               |2         |
|Testcricket        |2         |
|kkr                |1         |
+-------------------+----------+



### 13. Comments per subreddit

Check comment counts in the filtered comments.

In [11]:
# Comment counts per subreddit
verify_coms.groupBy("subreddit") \
    .agg(count("*").alias("comment_count")) \
    .orderBy(col("comment_count").desc()) \
    .show(50, truncate=False)


+-------------------+-------------+
|subreddit          |comment_count|
+-------------------+-------------+
|Cricket            |2191850      |
|CricketShitpost    |245296       |
|IndiaCricket       |218801       |
|ipl                |163245       |
|HiTMAN             |97158        |
|PakCricket         |80531        |
|RCB                |58438        |
|MumbaiIndians      |35785        |
|SunrisersHyderabad |31681        |
|KolkataKnightRiders|28898        |
|csk                |27100        |
|delhicapitals      |21713        |
|RajasthanRoyals    |12600        |
|EnglandCricket     |3510         |
|ViratKohli         |769          |
|RohitSharma        |177          |
|srh                |6            |
|Dhoni              |5            |
|IndianCricket      |5            |
|BCCI               |1            |
|AusCricket         |1            |
|lsg                |1            |
+-------------------+-------------+



### 14. Stop Spark

Free up resources.

In [12]:
# Stop Spark
spark.stop()
